# Running AgenticDataBench with fabric-rlm

[AgenticDataBench](https://github.com/AgenticDataBench/AgenticDataBench) gives an agent
one or more real data files and a business question, and grades the **file** it writes
against a gold file. No LLM judge: scoring is `compare_csv` and friends, with partial
credit per matching column.

This notebook runs it end to end on a handful of tasks. It is the same harness used for
the full 246-task runs, reduced to what fits comfortably in a notebook.

**No fabric-rlm changes are needed.** Everything below uses the public API of the
released package: `RLM.task(...)` plus four documented keyword arguments
(`max_turns`, `timeout`, `reserve_finalize_turns`, `output_validator`).

What you do need is a small adapter, because the benchmark and the library disagree
about where an answer lives: the benchmark grades a file on disk, the library returns a
payload. The adapter below bridges that in about 40 lines.

In [ ]:
%pip install -q fabric-rlm
# Libraries the model's generated code reaches for on this benchmark. Without them
# tasks die on ModuleNotFoundError, and the sandbox cannot pip install for itself.
%pip install -q statsmodels geopandas dbfread lightgbm pyshp jsonlines tqdm fuzzywuzzy

## 1. Configuration

In a Fabric notebook `FabricLM` uses the capacity's built-in endpoint, so there is no key
to manage. Off Fabric, set `OPENAI_API_KEY` and use `OpenAILM`, or point `dspy` at any
other provider.

In [ ]:
from pathlib import Path

# WHERE EVERYTHING IS WRITTEN.
#
# `/lakehouse/default/Files` is the mount for the notebook's attached default
# Lakehouse. It is a symlink: its real target lives under /synfs/..., so any path
# printed after Path.resolve() shows the /synfs/ form. Both spellings are the same
# location, and files written there are in the Lakehouse -- you can download them
# from the Lakehouse explorer.
#
# If NO Lakehouse is attached the mount is absent and the fallback below is session
# storage, which is deleted when the session ends. The check is whether the mount
# exists, not what the resolved path looks like.
#
# To use a non-default Lakehouse, set the path explicitly, e.g.
#   WORK = Path("/lakehouse/MyLakehouse/Files/agenticdatabench")
LAKEHOUSE = Path("/lakehouse/default/Files")

if LAKEHOUSE.exists():
    WORK, PERSISTENT = LAKEHOUSE / "agenticdatabench", True
else:
    WORK, PERSISTENT = Path.cwd() / "agenticdatabench", False

WORK.mkdir(parents=True, exist_ok=True)
BENCH = WORK / "AgenticDataBench"
TESTBED = BENCH / "testbed"
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)

# Start small. Each of these needs one modest CSV.
TASK_IDS = ["tourism_32", "tourism_25", "social_network_18"]

print(f"working directory : {WORK}")
print(f"  resolves to     : {WORK.resolve()}")
print(f"task outputs      : {RESULTS}/<task_id>/")
if PERSISTENT:
    print("\nPERSISTENT: this is the attached Lakehouse. Outputs appear under")
    print("Files/agenticdatabench/results/ in the Lakehouse explorer (hit refresh).")
    print("A /synfs/... resolved path above is normal -- it is the mount's real target.")
else:
    print("\nWARNING: no Lakehouse mounted at /lakehouse/default/Files.")
    print("Writing to SESSION-LOCAL storage, which is deleted when the session ends.")
    print("Either attach a Lakehouse (Explorer pane, left) and re-run this cell,")
    print("or run the final 'copy results to a Lakehouse' cell before you finish.")

In [ ]:
from fabric_rlm import RLM, File

def make_lm():
    """FabricLM on a Fabric capacity, OpenAILM otherwise."""
    try:
        from fabric_rlm import FabricLM
        return FabricLM("gpt-5.1")
    except Exception:
        from fabric_rlm import OpenAILM
        return OpenAILM("gpt-4o-mini")

LM = make_lm()
print("LM ready:", type(LM).__name__)

## 2. Fetch the benchmark and just the data these tasks need

The full dataset is about 27 GB, so we download only the files our chosen tasks
reference. Each task lists them in `data_sources`.

In [ ]:
import json, subprocess, urllib.parse, urllib.request

if not TESTBED.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/AgenticDataBench/AgenticDataBench.git", str(BENCH)],
                   check=True)

TASKS = {json.loads(l)["id"]: json.loads(l)
         for l in open(TESTBED / "tasks" / "dev.jsonl", encoding="utf-8")}
print(f"{len(TASKS)} public tasks available")

HF = "https://huggingface.co/datasets/shawnzzzh/AgenticDataBench/resolve/main/"

def fetch(task):
    """Download this task's inputs into testbed/datasets/<domain>/."""
    for src in task.get("data_sources", []):
        rel = src
        for prefix in (f"datasets/{task['domain']}/", f"{task['domain']}/"):
            if rel.startswith(prefix):
                rel = rel[len(prefix):]
        dest = TESTBED / "datasets" / task["domain"] / rel
        if dest.exists():
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = HF + urllib.parse.quote(f"datasets/{task['domain']}/{rel}")
        print("downloading", rel)
        with urllib.request.urlopen(url, timeout=1800) as r, open(dest, "wb") as fh:
            while chunk := r.read(1 << 20):
                fh.write(chunk)

for tid in TASK_IDS:
    fetch(TASKS[tid])
print("inputs ready")

## 3. The adapter

One benchmark task becomes one `RLM.task(...)` call. Three details matter, all learned
from failure analysis of the full runs:

- **Say the exact output filenames and directory.** The grader looks for
  `<outdir>/<task_id>/output.csv` (or whatever the task declares). Tasks that wrote a
  correctly computed file under the wrong name scored zero.
- **Validate before accepting SUBMIT.** `output_validator` raises `AssertionError` when
  the required file is missing or empty, and the runtime turns that into a repair turn.
  On the full run this lifted the completion rate from 89% to 96%.
- **Reserve turns to finalize.** Without it, tasks that run long simply end with nothing
  on disk.

In [ ]:
import re, time

def as_list(v):
    return v if isinstance(v, list) else [v]

def make_validator(task_dir, names):
    def validate(payload):
        missing = [n for n in names if not (task_dir / n).exists()]
        assert not missing, (
            f"required output file(s) not written: {missing}. Write them to "
            f"{task_dir} with exactly those names, then SUBMIT again.")
        for n in names:
            assert (task_dir / n).stat().st_size > 0, f"{n} is empty."
    return validate

def run_task(tid, outdir=RESULTS, max_turns=25, timeout=1800):
    task = TASKS[tid]
    task_dir = outdir / tid
    task_dir.mkdir(parents=True, exist_ok=True)

    inputs, lines = {}, []
    for i, src in enumerate(task.get("data_sources", [])):
        rel = src
        for prefix in (f"datasets/{task['domain']}/", f"{task['domain']}/"):
            if rel.startswith(prefix):
                rel = rel[len(prefix):]
        path = TESTBED / "datasets" / task["domain"] / rel
        key = re.sub(r"\W+", "_", Path(src).stem).strip("_").lower() or f"file_{i}"
        inputs[key] = File(path)
        lines.append(f"- `{key}`: {src}")

    outs = as_list(task["output_file_name"])
    prompt = (
        f"{task['question']}\n\n"
        f"Input files provided as File handles:\n" + "\n".join(lines) + "\n\n"
        f"Write the required output file(s) ({', '.join(outs)}) with exactly those "
        f"file names into this directory: {task_dir.resolve()}\n"
        f"After writing, reload each output file and check its columns before SUBMIT."
    )

    started = time.monotonic()
    result = RLM.task(
        task=prompt,
        inputs=inputs,
        outputs=["summary"],
        lm=LM,
        max_turns=max_turns,
        timeout=timeout,
        reserve_finalize_turns=2,
        output_validator=make_validator(task_dir, outs),
    ).run()

    return {
        "id": tid,
        "seconds": round(time.monotonic() - started, 1),
        "wrote": [n for n in outs if (task_dir / n).exists()],
        "summary": str(getattr(result, "summary", ""))[:300],
    }

## 4. Run

In [ ]:
records = []
for tid in TASK_IDS:
    print(f"running {tid} ...")
    try:
        rec = run_task(tid)
    except Exception as exc:                 # one bad task must not stop the batch
        rec = {"id": tid, "error": f"{type(exc).__name__}: {exc}"}
    records.append(rec)
    print("  ", rec)

## 5. Grade with the benchmark's own comparators

Each task carries an `eval_func` string such as
`compare_csv('output.csv', 'result.csv', ignore_order=True)`. We substitute real paths
and evaluate it against `da_agent.evaluators.metrics`, so the scores come from the
benchmark's code, not ours.

Two notes if you compare against published numbers:

- The paper's leaderboard is **n=344** (these 246 public tasks plus 98 withheld private
  ones). Scores here cover the public subset only and are not directly comparable.
- `evaluate.py` shipped with the benchmark fails on Windows, because it substitutes
  Windows paths into a regex replacement template and then `eval()`s them as string
  literals. The loop below sidesteps both by substituting `repr(path)`; it was verified
  to reproduce the stock evaluator's scores exactly, task for task, on Linux-style paths.

In [ ]:
import sys
sys.path.insert(0, str(TESTBED))
from da_agent.evaluators import metrics

def grade(tid, outdir=RESULTS):
    task = TASKS[tid]
    task_dir, gold_dir = outdir / tid, TESTBED / "gold" / tid
    scores = []
    for func in as_list(task["eval_func"]):
        exe = func
        for name in as_list(task["gold_file_name"]):
            lit = repr(str(gold_dir / Path(name).name))
            exe = re.sub(rf"(['\"]){re.escape(name)}\1", lambda m, p=lit: p, exe)
        for name in as_list(task["output_file_name"]):
            lit = repr(str(task_dir / Path(name).name))
            exe = re.sub(rf"(['\"]){re.escape(name)}\1", lambda m, p=lit: p, exe)
        try:
            out = eval(exe, {fn: getattr(metrics, fn) for fn in dir(metrics)})
            scores.append(out.get("score", 0.0) if isinstance(out, dict) else out)
        except Exception as exc:
            print(f"  {tid}: {type(exc).__name__}: {exc}")
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0

graded = {tid: grade(tid) for tid in TASK_IDS}
for tid, s in graded.items():
    print(f"{tid:20s} {s:.3f}")
print(f"\nmean: {sum(graded.values()) / len(graded):.3f}")

## 6. Keep the results (only needed if you ran without a Lakehouse attached)

If the configuration cell printed the WARNING, the outputs are in session storage and go
away when the session ends. This copies them into a Lakehouse.

`notebookutils.fs.cp` works even when the target Lakehouse is not the attached default —
give it an `abfss://` URL from **Copy ABFS path** in the Lakehouse explorer. If the
Lakehouse *is* mounted, a plain filesystem copy is enough.

In [ ]:
import shutil

# EITHER: the mounted default Lakehouse (leave DEST_ABFSS = None)
# OR:     any Lakehouse by ABFS path, e.g.
#         "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/adb_results"
DEST_ABFSS = None

if PERSISTENT and DEST_ABFSS is None:
    print("Already writing into the attached Lakehouse; nothing to copy.")
elif DEST_ABFSS:
    import notebookutils
    notebookutils.fs.cp(f"file://{RESULTS}", DEST_ABFSS, recurse=True)
    print("copied to", DEST_ABFSS)
    print(notebookutils.fs.ls(DEST_ABFSS))
else:
    dest = Path("/lakehouse/default/Files/adb_results")
    if not dest.parent.exists():
        raise SystemExit(
            "No Lakehouse mounted. Attach one in the Explorer pane, or set "
            "DEST_ABFSS to an abfss:// path from 'Copy ABFS path'.")
    dest.mkdir(parents=True, exist_ok=True)
    for task_dir in sorted(p for p in RESULTS.iterdir() if p.is_dir()):
        shutil.copytree(task_dir, dest / task_dir.name, dirs_exist_ok=True)
        print("copied", task_dir.name)
    print("\nnow in the Lakehouse under Files/adb_results/")

## 6. How the full run was done, and what mattered

The 246-task runs used exactly the adapter above, sharded six ways for parallelism, with
`--temperature 0`, `max_turns=25` and `timeout=1800`.

**Scores are fractional, not pass/fail.** `compare_csv` awards partial credit per
matching column, so a task can score 0.6 by getting three of five columns right.

**Measured on the full public set (MiniMax M3):**

| configuration | score |
|---|---|
| plain `RLM.task` | 43.49% |
| plus validator, reserved finalize turns, longer timeout, missing libraries installed | **47.98%** |

The gain (+4.5 points, 95% CI [+0.3, +8.7]) came almost entirely from tasks that
previously wrote **no file at all**: 32% of all zero scores. Completion went from 89% to
96%.

**Things that were tried and did not work**, each measured against the same baseline on a
stratified 40-task sample:

| change | score effect | token cost |
|---|---|---|
| inject schema previews of every input | -0.04 | **+44%** |
| enable skill autoloading and the router | -0.07 | **+163%** |
| blind double-solve with reconciliation (`verified_task` style) | -0.04 | **+185%** |

They share a cause worth remembering when tuning any RLM: the full transcript is resent
every turn, so anything added to the prompt is paid for roughly once per turn. On these
tasks input tokens were already 96% of volume, and a 14 KB playbook cost more than the
discovery turns it saved.

The double-solve ensemble failed for a second reason: two blind attempts produced
byte-comparable files only 15% of the time, so agreement carried no signal and nearly
every task paid for a third reconciliation run. That technique earns its keep when the
answer is a short determinate value, not an eight-column table of computed floats.

**If you want to push further**, the remaining headroom is in correctness rather than
mechanics: of the tasks still scoring zero, roughly 29% produce a file whose values are
wrong and 16% get the row count wrong.